### **1. Importación y Limpieza del carácter \\:** 
Importarmos el csv obtenido de la app, y limpiamos de las columnas y filas el caracter \\.

In [1]:
import pandas as pd

columnas_apoyo = [f'Col_{i}' for i in range(30)]

df = pd.read_csv('rent_features.csv', header=None, names=columnas_apoyo)

df = df.dropna(how='all', axis=1)

encabezados_originales = list(df.iloc[0])

total_columnas_reales = df.shape[1]

encabezados_limpios = [str(col).rstrip('\\') if pd.notna(col) else col for col in encabezados_originales]

nuevos_nombres = []
for i in range(total_columnas_reales):
    if i < len(encabezados_limpios) and pd.notna(encabezados_originales[i]):
        nuevos_nombres.append(encabezados_limpios[i])
    else:
        nuevos_nombres.append(f'Dato_Extra_{i+1}')

df.columns = nuevos_nombres
df = df.drop(0).reset_index(drop=True)

columnas_a_limpiar = [col for col in df.columns if not str(col).startswith('Dato_Extra_')]

for col in columnas_a_limpiar:
    df[col] = df[col].astype(str).str.rstrip('\\').replace('nan', pd.NA)


### **2. Limpieza y formateo de la direccion:** 
Unimos todo el registro de dirección para que sea uno solo y lo pasamos a minúsculas.

In [ ]:

df['direccion'] = df['direccion'].astype(str).str.lower().str.strip()

columnas_extra_formatear = [col for col in df.columns if str(col).startswith('Dato_Extra_')]
for col in columnas_extra_formatear:
    df[col] = df[col].astype(str).str.lower().str.strip()

for col in columnas_extra_formatear:
    df['direccion'] += ', ' + df[col]

columnas_a_eliminar = [col for col in df.columns if str(col).startswith('Dato_Extra_')]
df = df.drop(columns=columnas_a_eliminar)

### **3. Limpieza y asignación de Nones:** 
Limpiamos los nones del ETL asignando 0 a los valores que puede entregarnos None el csv.

In [ ]:
df.loc[df['m2_terreno'] == 'None', 'm2_terreno'] = df['m2_construccion']

df.loc[df['amen_Elevador'] == 'None', 'amen_Elevador'] = 0

### **4. Revisar columnas numericas :** 
Limpiar datos numericos y eliminar si quedaron algunos nulos.

In [ ]:
columnas_a_convertir = df.columns[:-2]

for col in columnas_a_convertir:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['tipo_propiedad_original'] = df['tipo_propiedad_original'].astype('string')
df['direccion'] = df['direccion'].astype('string')

### **5. Pasar a un csv limpio el dataset:** 
Pasamos el csv que limpiamos a limpio.

In [11]:
df.to_csv('inmuebles_limpios.csv', index=False, encoding='utf-8')